In [0]:
import mlflow
import pandas as pd
from pyspark.sql import functions as F

mlflow.set_registry_uri("databricks-uc")
CATALOG, SCHEMA, VOLUME = "workspace", "default", "opssat-ad"
NOM_MODELE = f"{CATALOG}.{SCHEMA}.opssat_anomaly_xgb"
T_PRED = f"{CATALOG}.{SCHEMA}.predictions_anomalies"
SEUIL = 0.5

PEAKS = ["n_peaks", "smooth20_n_peaks", "diff_peaks", "diff2_peaks"]
FEATURES = ["duration", "len", "mean", "var", "std", "kurtosis", "skew",
            "n_peaks", "smooth10_n_peaks", "smooth20_n_peaks", "diff_peaks", "diff2_peaks",
            "diff_var", "diff2_var", "gaps_squared", "len_weighted",
            "var_div_duration", "var_div_len"]
EXTRA = ["min", "max", "amplitude", "mediane", "iqr", "saut_max", "saut_moyen",
         "changements_sens", "pente", "autocorr_1", "trou_max"]

In [0]:
g = spark.table(f"{CATALOG}.{SCHEMA}.gold_segment_features")
extra = spark.table(f"{CATALOG}.{SCHEMA}.gold_features_extra")
ref = (spark.read.option("header", True).option("inferSchema", True)
       .csv(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/dataset.csv"))

donnees = (g.drop(*PEAKS)
           .join(ref.select("segment", *PEAKS), "segment")
           .join(extra, "segment"))
pdf = donnees.toPandas()

In [0]:
%pip install xgboost

In [0]:
modele = mlflow.xgboost.load_model(f"models:/{NOM_MODELE}@champion")
pdf["proba_anomalie"] = modele.predict_proba(pdf[FEATURES + EXTRA])[:, 1]
pdf["alerte"] = (pdf["proba_anomalie"] >= SEUIL).astype(int)

In [0]:
version = mlflow.MlflowClient().get_model_version_by_alias(NOM_MODELE, "champion").version

predictions = (
    spark.createDataFrame(pdf[["segment", "channel", "train", "anomaly", "proba_anomalie", "alerte"]])
    .withColumn("seuil", F.lit(SEUIL))
    .withColumn("version_modele", F.lit(version))
    .withColumn("date_scoring", F.current_timestamp())
)
predictions.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T_PRED)
display(spark.table(T_PRED).groupBy("channel").agg(
    F.count("*").alias("segments"), F.sum("alerte").alias("alertes")).orderBy("channel"))